# 👉 Урок 6 & 7. Векторные базы данных и чанкинг документов


## 🎯 Цели урока

- вспомнить зачем нужны базы данных;
- понять что такое векторная база данных;
- познакмиться с векторной БД FAISS;
- познакомиться с техникой чанкинга документов;
- построить retrieval слой RAG пайплайна.

---


Автор курса: Логинов Дмитрий Владимирович, преподаватель и методист МШП. Исследователь в области машинного обучения.



# 🗄️ Часть 1. Что такое база данных?

# Проблема хранения данных

Представим интернет-магазин.

Для каждого товара нужно хранить:

- название;
- цену;
- описание;
- количество на складе.

Например:

| id | Название | Цена |
|----|-----------|-------|
| 1 | Ноутбук | 70000 |
| 2 | Мышь | 1500 |
| 3 | Клавиатура | 5000 |

---

Если товаров:

```text
10
```

можно хранить их в Excel.

---

Если товаров:

```text
10 000 000
```

Excel уже не подходит.

---

Возникают задачи:

- быстро найти запись;
- быстро изменить запись;
- хранить данные без потерь;
- работать одновременно тысячам пользователей.

---

# Определение

База данных (Database) — организованное хранилище информации,
которое позволяет эффективно сохранять,
изменять и искать данные.

---

# Что делает база данных?

Типичные операции:

## Добавление данных

```text
Добавить нового пользователя
```

---

## Поиск данных

```text
Найти пользователя по id
```

---

## Изменение данных

```text
Изменить цену товара
```

---

## Удаление данных

```text
Удалить запись
```

---

Эти четыре операции называют:

## CRUD

Create
Read
Update
Delete

# 🏛️ Часть 2. Реляционные базы данных

### Таблица пользователей

| id | Имя | Возраст |
|----|------|----------|
| 1 | Анна | 15 |
| 2 | Иван | 16 |
| 3 | Мария | 17 |

---

Каждая строка называется:

## Запись (Record)

---

Каждый столбец называется:

## Поле (Field)

---

# SQL

Для работы с базой используется язык SQL.

SQL означает:

Structured Query Language

---

Например:

Найти всех пользователей старше 15 лет.

```sql
SELECT *
FROM users
WHERE age > 15;
```

---

# Почему это работает быстро?

Представим:

Таблица содержит

```text
100 000 000 строк
```

---

Как найти нужного пользователя?

Можно проверять каждую строку.

Но это долго.

---

Поэтому используются:

# Индексы

Индекс работает примерно так же,
как оглавление в книге.

---

Вместо просмотра каждой страницы
мы сразу переходим в нужное место.

---

Именно индексы позволяют выполнять поиск
за миллисекунды.

# 🤖 Часть 3. Почему обычные БД плохо подходят для Embeddings?

---

На прошлом уроке мы изучили embeddings.

Напомним:

Embedding — это вектор.

Например:

```python
[0.23, -0.51, 1.12, ...]
```

---

Современные модели используют:

- 384 измерения
- 768 измерений
- 1024 измерения
- 1536 измерений

---

Каждый документ превращается в такой вектор.

---

# Новая задача

Представим базу данных документов.

Документ №1:

```text
Как приготовить пасту
```

---

Документ №2:

```text
Рецепт итальянских макарон
```

---

После векторизации:

```python
doc1 = [0.23, 0.44, ...]
doc2 = [0.21, 0.47, ...]
```

---

# Что хочет пользователь?

Он больше не спрашивает:

```sql
WHERE id = 15
```

---

Он хочет:

```text
Найди наиболее похожий документ
```

---

Именно здесь обычная база данных начинает испытывать трудности.

---

# Почему?

Потому что SQL отлично работает с операциями:

```text
=
>
<
```

---

Но плохо работает с операцией:

```text
Найди ближайший вектор
```

---

Это принципиально другая задача.

# 🚀 Часть 4. Векторные базы данных и FAISS

---

# Новая задача

У нас есть:

```text
10 миллионов документов
```

---

Каждый документ представлен embedding-вектором.

Например:

```python
[0.21, -0.83, ...]
```

---

Когда пользователь вводит запрос:

```text
Как приготовить пасту?
```

мы тоже превращаем его в embedding.

---

После этого необходимо найти:

## ближайшие вектора

---

# Самый простой алгоритм

Сравнить запрос со всеми документами.

---

Для миллиона документов:

```text
1 000 000 сравнений
```

---

Для ста миллионов:

```text
100 000 000 сравнений
```

---

Поиск становится слишком медленным.

---

# Что делает FAISS?

FAISS решает задачу:

> Как находить ближайшие вектора очень быстро?

---

FAISS означает:

Facebook AI Similarity Search

---

Создан исследователями компании Meta.

---

# Основная идея

Вместо полного перебора всех документов

FAISS строит специальный индекс.

---

После построения индекса поиск выполняется
в тысячи раз быстрее.

---

# Как работает поиск в FAISS?

Шаг 1.

Сохраняем embeddings.

```text
Документ → Embedding
```

---

Шаг 2.

Строим индекс.

```text
Embeddings → FAISS Index
```

---

Шаг 3.

Получаем запрос пользователя.

```text
Запрос → Embedding
```

---

Шаг 4.

Ищем ближайшие вектора.

---

Шаг 5.

Возвращаем соответствующие документы.

---

# Где используется FAISS?

Практически в любой современной системе поиска:

- поиск по документам;
- поиск по PDF;
- рекомендательные системы;
- Retrieval-Augmented Generation (RAG);
- AI-ассистенты;
- интеллектуальные поисковые системы.

---

# Главный вывод первого урока

Реляционные БД отвечают на вопрос:

```text
Как найти запись с определённым значением?
```

---

Векторные БД отвечают на вопрос:

```text
Как найти наиболее похожий объект?
```

---

Появление embeddings привело к возникновению нового класса систем хранения данных — векторных баз данных.

#🔗 Часть 5. Оркестрация. Зачем нужен LangChain?

# Как выглядит обычная программа?

Представим программу-калькулятор.

```text
Ввод
↓
Вычисление
↓
Ответ
```

Выглядит достаточно просто, не так ли?
---

# А как выглядит современный AI-сервис в продакшене?

Продакшн (от англ. production — производство) — это готовое IT решение сложной задачи.

Представим ChatGPT с доступом к документам.

---

Чтобы ответить на вопрос пользователя,
нужно выполнить множество шагов.

```text
Запрос пользователя
↓
Поиск документов
↓
Поиск нужных фрагментов
↓
Подготовка контекста
↓
Отправка в LLM
↓
Ответ
```

---

Получается уже не одна операция,
а целая цепочка действий.

---

# Возникает проблема

Если писать всё вручную:

```python
load_document()

chunk_document()

create_embeddings()

save_to_faiss()

search()

call_llm()
```

---

Код быстро становится сложным.

---

Особенно когда появляется:

- несколько документов;
- несколько моделей;
- поиск;
- память;
- агенты.

---

# Решение

Нужен инструмент,
который поможет собирать AI-системы из готовых блоков.

# Что такое LangChain?

LangChain — это фреймворк для создания приложений на основе языковых моделей.

---

Если очень упростить:

```text
TensorFlow или PyTorch помогает строить нейросети

↓

LangChain помогает строить LLM-системы
```

---

# Главная идея

Любое AI-приложение можно представить как набор шагов.

Например:

```text
Документ
↓
Чанкинг
↓
Embeddings
↓
FAISS
↓
Поиск
```

---

LangChain позволяет собрать этот процесс
из готовых компонентов.

---

# Какие компоненты есть в LangChain?

Самые важные для нас:

---

## Document Loader

Загружает документы.

---

Например:

```text
PDF
TXT
DOCX
HTML
```

---

## Text Splitter

Разбивает текст на чанки.

---

## Embedding Model

Создает embeddings.

---

## Vector Store

Хранит embeddings.

---

Например:

```text
FAISS
Chroma
Pinecone
```

---

## Retriever

Выполняет поиск.

---

# Визуально

```text
Document Loader
         ↓

    Text Splitter
         ↓

  Embedding Model
         ↓

    Vector Store
         ↓

      Retriever
```

---

Это и есть простейший Retrieval Pipeline.

---

# Главный вывод

LangChain не заменяет:

- embeddings;
- FAISS;
- LLM.

---

Он помогает связать их вместе в единую систему.

# ✂️ Часть 6. Почему нужны чанки?

# Представим документ

Статья:

```text
История искусственного интеллекта
```

---

Размер:

```text
20 страниц
```

---

Пользователь спрашивает:

```text
Кто предложил термин Artificial Intelligence?
```

---

# Первый вариант

На самом деле embedding можно создавать не только у отдельных токенов. Можно создать embedding слова и даже всей статьи.

---

Получится:

```text
20 страниц
↓
1 embedding
```

---

Кажется удобно.

Но возникает проблема.

---

# Что хранит embedding?

Embedding хранит общий смысл текста.

---

Если текст небольшой - это хорошо.

Если текст огромный:

embedding будет полностью бесполезным.

---

Например:

Статья одновременно рассказывает:

- про Тьюринга;
- про Маккарти;
- про нейросети;
- про ChatGPT.

---

В результате embedding становится:

```text
"средним смыслом"
```

всей статьи.

---

# Почему это плохо?

Запрос:

```text
Кто ввёл термин Artificial Intelligence?
```

касается буквально нескольких предложений.

Но поиск будет сравнивать запрос с embedding всей статьи. Получается много лишней информации.

# Решение

Разделить документ на части. Этот процесс называется - Chunking